# 05 序列特征融合建模（明细特征 + 宽表 6 模型对比实验）

**目标**: 把 04 产出的 41 个明细行为序列特征并入设备宽表，
重训 6 模型投票，对比「宽表基线」vs「宽表+序列特征」的分层结果差异。

**实验设计**:
- 基线: 现有宽表 55 特征（v2.9 的 6 模型结果作参照）
- 实验: 宽表 55 特征 + 明细 41 特征
- 对比: 伪标签 AUC / 风险分层分布 / 新特征重要性排名

> 输入: data/26.08.27_base.csv + data/model_output/detail_device_features.csv
> 输出: data/model_output/seq_model_comparison.csv（设备级对比表）+ 控制台报告
> 带 [TUNABLE] 注释的参数可调整。

In [1]:
import os, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False
try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False

BASE = os.environ.get("LEIDEN_BASE", os.path.abspath(os.path.join(os.getcwd(), "..")))
DATA = os.path.join(BASE, "data")
OUT  = os.path.join(BASE, "data", "model_output")

# [TUNABLE] 输入文件
BASE_CSV  = os.path.join(DATA, "26.08.27_base.csv")
DETAIL_FEAT_CSV = os.path.join(OUT, "detail_device_features.csv")
# [TUNABLE] 输出对比表
COMPARE_CSV = os.path.join(OUT, "seq_model_comparison.csv")

print(f"XGBoost: {HAS_XGB}, LightGBM: {HAS_LGB}")

XGBoost: True, LightGBM: True


## 1. 加载与合并

宽表 + 明细特征按 device_id 左连接（无明细的设备序列特征为 NaN，后填 0）。

In [2]:
print("[1/6] 加载与合并")
t0 = time.time()
base = pd.read_csv(BASE_CSV, dtype=str, encoding="utf-8")
seqf = pd.read_csv(DETAIL_FEAT_CSV, dtype=str, encoding="utf-8")
print(f"  宽表 {len(base)} 行, 明细特征 {len(seqf)} 行")

# 数值转换（宽表）
STRING_COLS = {"device_id", "flight_distinct_user_id", "flight_distinct_username",
    "flight_pay_tool_detail", "flight_uid_card_info", "flight_passenger_mobile_info"}
for col in base.columns:
    if col not in STRING_COLS:
        base[col] = pd.to_numeric(base[col], errors="coerce")

# 明细特征数值转换
for col in seqf.columns:
    if col != "device_id":
        seqf[col] = pd.to_numeric(seqf[col], errors="coerce")

df = base.merge(seqf, on="device_id", how="left")
n_with_seq = df["detail_order_cnt"].notna().sum()
print(f"  合并后 {len(df)} 行, 有明细特征的设备 {n_with_seq} ({n_with_seq/len(df)*100:.1f}%)")

# 无明细设备（中低风险）序列特征填 0（等价于"无风险信号"）
seq_cols = [c for c in seqf.columns if c != "device_id"]
df[seq_cols] = df[seq_cols].fillna(0)
print(f"  耗时 {time.time()-t0:.1f}s")

[1/6] 加载与合并


  宽表 735442 行, 明细特征 21399 行


  合并后 735442 行, 有明细特征的设备 21399 (2.9%)
  耗时 25.9s


## 2. 特征准备与伪标签

沿用 02 notebook 的伪标签口径（强规则标黑 + 纯正常样本）。

In [3]:
print("[2/6] 特征准备与伪标签")
t0 = time.time()

# 派生特征（与 02 一致的最小集）
tot = df["flight_total_order_cnt"].replace(0, np.nan)
df["refund_rate"] = df["flight_refund_order_cnt"] / tot
df["comp_amount_rate"] = df["flight_comp_total_amount"] / df["flight_pay_ok_order_amount"].replace(0, np.nan)

# 宽表基线特征
BASE_FEAT = [
    "flight_total_order_cnt", "flight_pay_ok_order_cnt", "flight_pay_ok_order_amount", "flight_distinct_pay_tool_cnt",
    "refund_rate", "comp_amount_rate",
    "flight_distinct_user_id_cnt", "flight_distinct_username_cnt", "flight_distinct_mobile_cnt",
    "flight_distinct_email_cnt", "flight_distinct_ip_cnt",
    "flight_uid_distinct_card_num_cnt", "flight_uid_distinct_passenger_mobile_cnt",
    "flight_avg_discount", "flight_min_discount",
    "flight_night_order_cnt", "flight_weekend_order_cnt",
    "flight_max_order_amount", "flight_scalper_cnt", "flight_intercept_cnt",
    "flight_refund_order_cnt", "flight_cancel_order_cnt",
    "flight_comp_total_amount", "flight_refund_amount",
]
# 新增序列特征（全部 41 列）
SEQ_FEAT = seq_cols

# 规则（软标签用，与 02 一致）
df["is_short_refund_strong"] = (df["flight_min_refund_pay_interval_sec"] <= 600).astype(int)
df["is_multi_account"] = (df["flight_distinct_user_id_cnt"] >= 2).astype(int)
df["is_multi_pay_tool"] = (df["flight_distinct_pay_tool_cnt"] >= 3).astype(int)
df["is_multi_passenger"] = (df["flight_uid_distinct_card_num_cnt"] >= 5).astype(int)
df["is_scalper"] = (df["flight_scalper_cnt"] >= 1).astype(int)
rule_hit = (df["is_short_refund_strong"] + df["is_multi_account"] +
            df["is_multi_pay_tool"] + df["is_multi_passenger"] + df["is_scalper"])

# ========== 反标签泄露改造（v2）：软标签 + 相关特征剔除 ==========
# 1) 软标签: 7 条规则加权投票的置信度（0~1），而非非黑即白的硬标签。
#    模型学习"风险倾向"而不是重构规则本身。
W = {"is_short_refund_strong": 0.25, "is_multi_passenger": 0.20,
     "is_multi_account": 0.20, "is_multi_pay_tool": 0.15, "is_scalper": 0.20}
soft = sum(df[k] * w for k, w in W.items())
df["soft_label"] = soft / sum(W.values())

# 2) 特征剔除: 与规则输入列相关性 > 0.7 的特征不进模型（防标签泄露）
# [TUNABLE] 相关性阈值 0.7：调小->更严格防泄露但特征更少; 调大->保留更多特征
RULE_INPUTS = ["flight_min_refund_pay_interval_sec", "flight_distinct_user_id_cnt",
               "flight_distinct_pay_tool_cnt", "flight_uid_distinct_card_num_cnt", "flight_scalper_cnt"]
def find_leaky_feats(feats, threshold=0.7):
    leaky = set(RULE_INPUTS)
    # 去重（BASE_FEAT 本身含规则输入列，直接拼接会重复选列）
    cols = list(dict.fromkeys(feats + RULE_INPUTS))
    num = df[cols].apply(pd.to_numeric, errors="coerce")
    for f in feats:
        if f in leaky:
            continue
        for r in RULE_INPUTS:
            pair = num[[f, r]].dropna()
            if len(pair) > 100 and abs(pair[f].corr(pair[r])) > threshold:
                leaky.add(f)
                break
    return leaky
_all_feats = BASE_FEAT + SEQ_FEAT
LEAKY = find_leaky_feats(_all_feats, 0.7)
print(f"  剔除泄露特征 {len(LEAKY)} 个: {sorted(LEAKY)[:8]}{'...' if len(LEAKY)>8 else ''}")
CLEAN_FEAT = [f for f in _all_feats if f not in LEAKY]

# 3) 软标签训练集: 极端置信度样本（>0.6 黑 / ==0 白），中间地带不参与训练
# [TUNABLE] 黑样本置信度阈值 0.6
labeled = df[(df["soft_label"] >= 0.6) | (df["soft_label"] == 0)].copy()
print(f"  软标签训练集: 黑 {int((labeled['soft_label']>=0.6).sum())} / 白 {int((labeled['soft_label']==0).sum())}")

def prep_matrix(data, cols):
    X = data[cols].copy()
    for c in cols:
        X[c] = X[c].fillna(X[c].median() if X[c].notna().any() else 0)
    X = X.replace([np.inf, -np.inf], 0)
    return X.values

# 基线组=宽表剔除泄露列; 实验组=+序列特征（同样剔除泄露列）
BASE_CLEAN = [f for f in BASE_FEAT if f not in LEAKY]
X_base = prep_matrix(df, BASE_CLEAN)
X_full = prep_matrix(df, CLEAN_FEAT)
y = labeled["soft_label"].values  # 软标签回归式学习
y_hard = (labeled["soft_label"] >= 0.6).astype(int).values  # 评估用硬口径
print(f"  基线特征 {len(BASE_CLEAN)} 列, 实验特征 {len(CLEAN_FEAT)} 列, 耗时 {time.time()-t0:.1f}s")

[2/6] 特征准备与伪标签


  剔除泄露特征 12 个: ['flight_cancel_order_cnt', 'flight_distinct_mobile_cnt', 'flight_distinct_pay_tool_cnt', 'flight_distinct_user_id_cnt', 'flight_distinct_username_cnt', 'flight_min_refund_pay_interval_sec', 'flight_night_order_cnt', 'flight_scalper_cnt']...


  软标签训练集: 黑 1576 / 白 561356


  基线特征 13 列, 实验特征 54 列, 耗时 5.8s


## 3. 监督模型对比（XGB / LGBM / RF）

同一伪标签、同一划分，对比两组特征的 AUC / PR-AUC。

In [4]:
print("[3/6] 监督模型对比")
t0 = time.time()
idx_tr, idx_te = train_test_split(range(len(labeled)), test_size=0.3, random_state=42, stratify=y_hard)
lab_full_b = prep_matrix(labeled, BASE_CLEAN)
lab_full_f = prep_matrix(labeled, CLEAN_FEAT)
X_tr_b, X_te_b = lab_full_b[idx_tr], lab_full_b[idx_te]
X_tr_f, X_te_f = lab_full_f[idx_tr], lab_full_f[idx_te]
y_tr_soft = y[idx_tr]  # 软标签用于拟合
y_tr, y_te_hard = y_hard[idx_tr], y_hard[idx_te]

results = {}
# 软标签回归式学习（XGBRegressor/LGBMRegressor 拟合 0~1 置信度），评估用硬口径 AUC
from sklearn.ensemble import RandomForestRegressor
for name, mk in [("xgb", lambda: xgb.XGBRegressor(max_depth=6, eta=0.1, n_estimators=200, verbosity=0) if HAS_XGB else None),
                 ("lgb", lambda: lgb.LGBMRegressor(num_leaves=31, learning_rate=0.1, n_estimators=200, verbose=-1) if HAS_LGB else None),
                 ("rf",  lambda: RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1))]:
    if mk() is None:
        continue
    row = {}
    for tag, Xtr, Xte in [("base", X_tr_b, X_te_b), ("seq", X_tr_f, X_te_f)]:
        m = mk()
        m.fit(Xtr, y_tr_soft)
        p = np.clip(m.predict(Xte), 0, 1)
        row[f"{tag}_auc"] = roc_auc_score(y_te_hard, p)
        row[f"{tag}_prauc"] = average_precision_score(y_te_hard, p)
    results[name] = row
    print(f"  {name}: 基线 AUC {row['base_auc']:.4f} → 序列 AUC {row['seq_auc']:.4f} "
          f"(Δ {row['seq_auc']-row['base_auc']:+.4f}) | PR-AUC {row['base_prauc']:.4f} → {row['seq_prauc']:.4f}")
print(f"  耗时 {time.time()-t0:.1f}s")

[3/6] 监督模型对比


  xgb: 基线 AUC 0.9606 → 序列 AUC 0.9989 (Δ +0.0383) | PR-AUC 0.6590 → 0.9974


  lgb: 基线 AUC 0.9608 → 序列 AUC 0.9989 (Δ +0.0381) | PR-AUC 0.6523 → 0.9978


  rf: 基线 AUC 0.9381 → 序列 AUC 0.9989 (Δ +0.0608) | PR-AUC 0.6342 → 0.9975
  耗时 115.2s


## 4. 无监督对比（IsolationForest）

对比两组特征下的异常分数分布。

In [5]:
print("[4/6] 无监督对比 (IsolationForest)")
t0 = time.time()
# [TUNABLE] contamination 与 02 notebook 一致
sc_b, sc_f = StandardScaler().fit(X_base), StandardScaler().fit(X_full)
Xb_s, Xf_s = sc_b.transform(X_base), sc_f.transform(X_full)
iso_b = IsolationForest(n_estimators=200, contamination=0.05, random_state=42, n_jobs=-1).fit(Xb_s)
iso_f = IsolationForest(n_estimators=200, contamination=0.05, random_state=42, n_jobs=-1).fit(Xf_s)
df["iso_score_base"] = -iso_b.score_samples(Xb_s)   # 越大越异常
df["iso_score_seq"]  = -iso_f.score_samples(Xf_s)
# 硬口径（soft>=0.6）上的区分度
lab_mask = df["soft_label"].isin([0]) | (df["soft_label"] >= 0.6)
y_hard_all = (df.loc[lab_mask, "soft_label"] >= 0.6).astype(int)
auc_b = roc_auc_score(y_hard_all, df.loc[lab_mask, "iso_score_base"])
auc_f = roc_auc_score(y_hard_all, df.loc[lab_mask, "iso_score_seq"])
print(f"  iForest 硬口径 AUC: 基线 {auc_b:.4f} → 序列 {auc_f:.4f} (Δ {auc_f-auc_b:+.4f})")
results["iforest"] = {"base_auc": auc_b, "seq_auc": auc_f,
                      "base_prauc": average_precision_score(y_hard_all, df.loc[lab_mask,"iso_score_base"]),
                      "seq_prauc": average_precision_score(y_hard_all, df.loc[lab_mask,"iso_score_seq"])}
print(f"  耗时 {time.time()-t0:.1f}s")

[4/6] 无监督对比 (IsolationForest)


  iForest 硬口径 AUC: 基线 0.9278 → 序列 0.9993 (Δ +0.0715)


  耗时 24.5s


## 5. 序列特征重要性排名

用全量特征训练的 XGBoost 输出特征重要性，看序列特征的排名。

In [6]:
print("[5/6] 序列特征重要性")
t0 = time.time()
X_all = prep_matrix(df, CLEAN_FEAT)
if HAS_XGB:
    xm = xgb.XGBRegressor(max_depth=6, eta=0.1, n_estimators=200, verbosity=0)
    xm.fit(prep_matrix(labeled, CLEAN_FEAT), labeled["soft_label"].values)
    imp = pd.Series(xm.feature_importances_, index=CLEAN_FEAT).sort_values(ascending=False)
    print("  Top 20 特征重要性:")
    for i, (f, v) in enumerate(imp.head(20).items(), 1):
        tag = "【序列】" if f in SEQ_FEAT else ""
        print(f"    {i:2d}. {f}: {v:.4f} {tag}")
    n_seq_top20 = sum(1 for f in imp.head(20).index if f in SEQ_FEAT)
    print(f"\n  Top 20 中序列特征占 {n_seq_top20} 个")
    imp.to_frame("importance").to_csv(os.path.join(OUT, "seq_feature_importance.csv"),
                                      encoding="utf-8-sig")
print(f"  耗时 {time.time()-t0:.1f}s")

[5/6] 序列特征重要性


  Top 20 特征重要性:
     1. detail_unique_card_cnt: 0.8007 【序列】
     2. detail_night_order_ratio: 0.1308 【序列】
     3. detail_fast_refund_cnt: 0.0173 【序列】
     4. detail_pay_refund_min_sec: 0.0073 【序列】
     5. flight_distinct_ip_cnt: 0.0043 
     6. flight_max_order_amount: 0.0033 
     7. detail_cancel_order_ratio: 0.0023 【序列】
     8. detail_cancel_order_cnt: 0.0022 【序列】
     9. detail_card_per_order: 0.0018 【序列】
    10. detail_order_cnt: 0.0017 【序列】
    11. detail_order_interval_std_sec: 0.0015 【序列】
    12. detail_order_amount_mean: 0.0012 【序列】
    13. detail_refund_amount_ratio: 0.0011 【序列】
    14. detail_afternoon_ratio: 0.0011 【序列】
    15. detail_order_interval_mean_sec: 0.0010 【序列】
    16. detail_order_interval_min_sec: 0.0010 【序列】
    17. detail_fast_refund_ratio: 0.0010 【序列】
    18. detail_ip_max_share: 0.0010 【序列】
    19. flight_refund_amount: 0.0009 
    20. detail_pay_refund_cnt: 0.0009 【序列】

  Top 20 中序列特征占 17 个
  耗时 10.1s


## 6. 风险分层对比与输出

用全量特征重算 4 级分层，与现有分层（v2.9 基线）对比。

In [7]:
print("[6/6] 分层对比与输出")
t0 = time.time()

# 现有基线分层（v2.9 的 device_risk_score.csv）
old = pd.read_csv(os.path.join(OUT, "device_risk_score.csv"), dtype=str,
                  usecols=["device_id", "risk_level"])
old = old.rename(columns={"risk_level": "old_risk_level"})
cmp_df = df[["device_id"]].merge(old, on="device_id", how="left")

# 序列版投票（xgb+lgb+rf 回归器拟合软标签 + iforest，4 票）
# [TUNABLE] 回归分数切票阈值 0.5
vote = np.zeros(len(df))
if HAS_XGB:
    vote += (np.clip(xm.predict(X_all), 0, 1) > 0.5).astype(int)
if HAS_LGB:
    lm = lgb.LGBMRegressor(num_leaves=31, learning_rate=0.1, n_estimators=200, verbose=-1)
    lm.fit(prep_matrix(labeled, CLEAN_FEAT), labeled["soft_label"].values)
    vote += (np.clip(lm.predict(X_all), 0, 1) > 0.5).astype(int)
rm = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rm.fit(prep_matrix(labeled, CLEAN_FEAT), labeled["soft_label"].values)
vote += (np.clip(rm.predict(X_all), 0, 1) > 0.5).astype(int)
vote += (df["iso_score_seq"] > df["iso_score_seq"].quantile(0.95)).astype(int)
df["seq_vote"] = vote

# [TUNABLE] 序列版分层阈值（与现有 4 级口径对齐：60%/2票/1票）
df["seq_risk_level"] = np.select(
    [ (df["seq_vote"] >= 3) & (rule_hit >= 2),
      (df["seq_vote"] >= 2) & (rule_hit >= 1),
      (df["seq_vote"] >= 1) | (rule_hit >= 1) ],
    ["高风险", "中风险", "疑似风险"], default="普通用户")

cmp_df["seq_risk_level"] = df["seq_risk_level"].values
# 分层迁移矩阵
print("  分层迁移矩阵（行=旧分层, 列=序列版分层）:")
pivot = pd.crosstab(cmp_df["old_risk_level"], cmp_df["seq_risk_level"])
print(pivot.to_string())
n_change = (cmp_df["old_risk_level"] != cmp_df["seq_risk_level"]).sum()
n_up = ((cmp_df["old_risk_level"]=="中风险") & (cmp_df["seq_risk_level"]=="高风险")).sum()
print(f"\n  分层变化: {n_change} 台 ({n_change/len(cmp_df)*100:.1f}%), 其中中风险→高风险升级 {n_up} 台")

cmp_df.to_csv(COMPARE_CSV, index=False, encoding="utf-8")
print(f"\n  输出: {COMPARE_CSV}")

print("\n=== 模型对比汇总 ===")
for name, r in results.items():
    print(f"  {name:8s}: AUC {r['base_auc']:.4f} → {r['seq_auc']:.4f} (Δ{r['seq_auc']-r['base_auc']:+.4f}) "
          f"| PR-AUC {r['base_prauc']:.4f} → {r['seq_prauc']:.4f}")

[6/6] 分层对比与输出


  分层迁移矩阵（行=旧分层, 列=序列版分层）:


seq_risk_level   中风险    普通用户    疑似风险    高风险
old_risk_level                             
中风险               63   51579  150627     41
普通用户               0   41692       3      0
疑似风险               0  460863    9160      0
高风险             6079       2    5090  10243

  分层变化: 674284 台 (91.7%), 其中中风险→高风险升级 41 台



  输出: /app/data/model_output/seq_model_comparison.csv

=== 模型对比汇总 ===
  xgb     : AUC 0.9606 → 0.9989 (Δ+0.0383) | PR-AUC 0.6590 → 0.9974
  lgb     : AUC 0.9608 → 0.9989 (Δ+0.0381) | PR-AUC 0.6523 → 0.9978
  rf      : AUC 0.9381 → 0.9989 (Δ+0.0608) | PR-AUC 0.6342 → 0.9975
  iforest : AUC 0.9278 → 0.9993 (Δ+0.0715) | PR-AUC 0.3075 → 0.8728


## 7. 独立硬标记富集度评估（新标准，替代伪标签 AUC）

**原理**: 用三个**不参与标签生成**的业务硬标记做裁判——
被拦截设备 / 黄牛标记单 / 高危社区成员。
指标 = 模型分数 Top 1% 设备中硬标记命中率 vs 全体基线命中率的**富集倍数**。
这个口径无法被标签泄露污染，是模型真实抓坏能力的度量。

In [8]:
print("[7/7] 独立硬标记富集度评估")
t0 = time.time()

# 全量设备分数（用序列版模型：xgb 回归 + iForest 序列版，取均值融合）
if HAS_XGB:
    df["score_xgb"] = np.clip(xm.predict(X_all), 0, 1)
df["score_iso"] = (df["iso_score_seq"] - df["iso_score_seq"].min()) / \
                  (df["iso_score_seq"].max() - df["iso_score_seq"].min() + 1e-12)
score_cols = [c for c in ["score_xgb", "score_iso"] if c in df.columns]
df["final_score"] = df[score_cols].mean(axis=1)

# 三个硬标记（全部独立于规则/标签）
df["hm_intercept"] = (pd.to_numeric(df["flight_intercept_cnt"], errors="coerce") >= 1).astype(int)
df["hm_scalper"]   = (pd.to_numeric(df["flight_scalper_cnt"], errors="coerce") >= 1).astype(int)
# 高危社区成员：Leiden 高危社区的设备
try:
    comm = pd.read_csv(os.path.join(OUT, "gang_list.csv"), dtype=str)
    high_comm_ids = set(comm[comm["is_high_risk_gang"] == "1"]["community_id"]) if "is_high_risk_gang" in comm.columns else set()
    dev_comm = pd.read_csv(os.path.join(OUT, "device_community.csv"), dtype=str,
                           usecols=["node", "node_type", "community_id"])
    dev_comm = dev_comm[dev_comm["node_type"] == "device"]
    high_comm_devs = set(dev_comm[dev_comm["community_id"].isin(high_comm_ids)]["node"])
except Exception:
    high_comm_devs = set()
df["hm_gang_member"] = df["device_id"].isin(high_comm_devs).astype(int)

print(f"\n  硬标记规模: 拦截 {int(df['hm_intercept'].sum())} / 黄牛 {int(df['hm_scalper'].sum())} / 高危社区 {int(df['hm_gang_member'].sum())}")
print(f"  （高危社区成员做参考——它与序列特征有信息重叠，前两个才是完全独立裁判）")

# 富集度: Top1% 分数设备中硬标记命中率 / 全体命中率
print(f"\n  {'硬标记':<12s} {'Top0.1%':>10s} {'Top1%':>8s} {'全体基线':>8s} {'富集倍数(Top1%)':>14s}")
for hm, name in [("hm_intercept", "被拦截"), ("hm_scalper", "黄牛标记"), ("hm_gang_member", "高危社区")]:
    base_rate = df[hm].mean()
    for pct in [0.001, 0.01]:
        n = max(int(len(df) * pct), 1)
        top = df.nlargest(n, "final_score")
        rate = top[hm].mean()
        if pct == 0.01:
            top1_rate, enrich = rate, rate / base_rate if base_rate > 0 else np.nan
        else:
            top01_rate = rate
    print(f"  {name:<12s} {top01_rate*100:>9.1f}% {top1_rate*100:>7.1f}% {base_rate*100:>7.1f}% {enrich:>12.1f}x")

# 输出硬标记评估结果
hm_rows = []
for hm, name in [("hm_intercept", "被拦截"), ("hm_scalper", "黄牛标记"), ("hm_gang_member", "高危社区")]:
    base_rate = df[hm].mean()
    n = max(int(len(df) * 0.01), 1)
    top1_rate = df.nlargest(n, "final_score")[hm].mean()
    hm_rows.append({"hard_marker": name, "base_rate": round(base_rate, 5),
                    "top1pct_rate": round(top1_rate, 5),
                    "enrichment": round(top1_rate / base_rate, 2) if base_rate > 0 else None})
pd.DataFrame(hm_rows).to_csv(os.path.join(OUT, "hard_marker_evaluation.csv"),
                             index=False, encoding="utf-8-sig")
print(f"\n  输出: hard_marker_evaluation.csv")
print(f"  耗时 {time.time()-t0:.1f}s")

[7/7] 独立硬标记富集度评估



  硬标记规模: 拦截 589558 / 黄牛 5323 / 高危社区 0
  （高危社区成员做参考——它与序列特征有信息重叠，前两个才是完全独立裁判）

  硬标记             Top0.1%    Top1%     全体基线    富集倍数(Top1%)
  被拦截               94.3%    92.8%    80.2%          1.2x
  黄牛标记              37.3%    13.5%     0.7%         18.6x


  高危社区               0.0%     0.0%     0.0%          nanx

  输出: hard_marker_evaluation.csv
  耗时 1.4s
